In [2]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (classification_report, roc_auc_score, 
                             average_precision_score, confusion_matrix)
from sklearn.preprocessing import label_binarize

df = pd.read_csv('../data/processed/dataset2_processed.csv')

X = df.drop(columns=['diagnosis'])
y = df['diagnosis'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [4]:
#models, eventually take care of the param. finetuning here, ask for feedback to sabrina
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM':           SVC(kernel='rbf', probability=True, random_state=42),
    'MLP':           MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42),
}

In [6]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    
    report = classification_report(y_test, y_pred, output_dict=True)
    auc_roc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
    y_bin = label_binarize(y_test, classes=[1, 2, 3])
    auc_pr = average_precision_score(y_bin, y_proba, average='macro')
    
    results.append({
        'Model': name,
        'Accuracy': round(report['accuracy'], 3),
        'Precision (macro)': round(report['macro avg']['precision'], 3),
        'Recall (macro)': round(report['macro avg']['recall'], 3),
        'F1 (macro)': round(report['macro avg']['f1-score'], 3),
        'AUC-ROC': round(auc_roc, 3),
        'AUC-PR': round(auc_pr, 3),
    })
    
    print(f"\n{'='*50}")
    print(f"  Model: {name}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=['Healthy','Benign','Cancer']))
    print(f"  AUC-ROC : {auc_roc:.3f}")
    print(f"  AUC-PR  : {auc_pr:.3f}")



  Model: Random Forest
              precision    recall  f1-score   support

     Healthy       0.96      1.00      0.98        51
      Benign       0.98      0.95      0.96        43
      Cancer       1.00      0.96      0.98        26

    accuracy                           0.97       120
   macro avg       0.98      0.97      0.98       120
weighted avg       0.98      0.97      0.97       120

  AUC-ROC : 0.995
  AUC-PR  : 0.993

  Model: SVM
              precision    recall  f1-score   support

     Healthy       0.92      0.92      0.92        51
      Benign       0.86      0.86      0.86        43
      Cancer       0.92      0.92      0.92        26

    accuracy                           0.90       120
   macro avg       0.90      0.90      0.90       120
weighted avg       0.90      0.90      0.90       120

  AUC-ROC : 0.983
  AUC-PR  : 0.967

  Model: MLP
              precision    recall  f1-score   support

     Healthy       0.93      0.84      0.89        51
     

In [12]:
summary = pd.DataFrame(results).set_index('Model')
print(summary.to_string())

               Accuracy  Precision (macro)  Recall (macro)  F1 (macro)  AUC-ROC  AUC-PR
Model                                                                                  
Random Forest     0.975              0.979           0.972       0.975    0.995   0.993
SVM               0.900              0.902           0.902       0.902    0.983   0.967
MLP               0.850              0.851           0.855       0.852    0.956   0.931


In [14]:
rf_model = models['Random Forest']
joblib.dump(rf_model,'../models/ds2/ds2_random_forest.pkl')
svm_model = models['SVM']
joblib.dump(svm_model,'../models/ds2/ds2_svm.pkl')
mlp_model = models['MLP']
joblib.dump(mlp_model,'../models/ds2/ds2_mlp.pkl')

np.save('../models/ds2/X_train.npy', X_train)
np.save('../models/ds2/X_test.npy', X_test)
np.save('../models/ds2/y_test.npy', y_test)
pd.Series(X.columns.tolist()).to_csv('../models/ds2/feature_names.csv', index=False)